# Peatland Data Cleaning

Notebook ini digunakan untuk melakukan data profiling, data quality
checking, validasi geometri, dan preprocessing data lahan gambut
Pulau Kalimantan.

Sumber data: BNPB - InAWARE / Global Forest Watch.

In [28]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from pathlib import Path

In [29]:
# Lokasi raw dataset
raw_path = Path("../../data/raw/peatland/peatland_kalimantan_bbox.geojson")

# Membaca GeoJSON
gdf_peatland = gpd.read_file(raw_path)

print("Data berhasil dibaca!")
print("Jumlah baris:", len(gdf_peatland))
print("Jumlah kolom:", len(gdf_peatland.columns))

Data berhasil dibaca!
Jumlah baris: 350
Jumlah kolom: 10


In [30]:
# Menampilkan 5 data pertama
display(gdf_peatland.head())

,Shape_Length,Shape_Area,OBJECTID_12,objectid_1,objectid,layer_revi,shape_leng,globalid,shape_Le_1,geometry
0,13663.737219,1.040841e+07,30,30,511,gambut,13507.658279,{E19D5C1A-D16F-453C-8FF6-28D8A171F6CF},13663.737219,"POLYGON ((112.61097 -3.38824, 112.61185 -3.386..."
1,14723.912350,1.082060e+07,47,47,528,gambut,14808.047964,{FB498FCA-744D-4E19-B5A2-454F64BBC207},14723.912350,"POLYGON ((112.56246 -3.29412, 112.56082 -3.292..."
2,10581.977876,7.189394e+06,57,57,538,gambut,10795.680213,{611B54C9-89D8-4CDC-A35D-1F582FF3AED1},10581.977876,"POLYGON ((114.25173 -3.22482, 114.25139 -3.224..."
3,19531.317319,1.438427e+07,60,60,541,gambut,20093.692704,{551286D6-F742-4BF2-A1BB-9FE2323290AB},19531.317319,"POLYGON ((114.23335 -3.22611, 114.23296 -3.226..."
4,17640.444164,1.299445e+07,66,66,547,gambut,17573.431145,{B681FC46-0847-48DF-B18B-16C44768260D},17640.444164,"POLYGON ((114.24329 -3.16534, 114.24329 -3.165..."


In [31]:
# Informasi struktur data
gdf_peatland.info()

print("\nNama kolom:")
print(gdf_peatland.columns.tolist())

print("\nCRS:")
print(gdf_peatland.crs)

print("\nTipe geometri:")
print(gdf_peatland.geom_type.value_counts())

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   Shape_Length  350 non-null    float64 
 1   Shape_Area    350 non-null    float64 
 2   OBJECTID_12   350 non-null    int32   
 3   objectid_1    350 non-null    int32   
 4   objectid      350 non-null    int32   
 5   layer_revi    350 non-null    str     
 6   shape_leng    350 non-null    float64 
 7   globalid      350 non-null    str     
 8   shape_Le_1    350 non-null    float64 
 9   geometry      350 non-null    geometry
dtypes: float64(4), geometry(1), int32(3), str(2)
memory usage: 38.4 KB

Nama kolom:
['Shape_Length', 'Shape_Area', 'OBJECTID_12', 'objectid_1', 'objectid', 'layer_revi', 'shape_leng', 'globalid', 'shape_Le_1', 'geometry']

CRS:
EPSG:4326

Tipe geometri:
Polygon         345
MultiPolygon      5
Name: count, dtype: int64


In [32]:
# Statistik atribut numerik
display(gdf_peatland.describe())

# Distribusi kategori lahan
print("\nDistribusi kategori layer_revi:")
print(gdf_peatland["layer_revi"].value_counts(dropna=False))

,Shape_Length,Shape_Area,OBJECTID_12,objectid_1,objectid,shape_leng,shape_Le_1
count,3.500000e+02,3.500000e+02,350.000000,350.000000,350.000000,3.500000e+02,3.500000e+02
mean,5.529812e+04,1.418179e+08,826.797143,826.797143,1307.797143,5.630060e+04,5.529812e+04
std,1.053407e+05,5.890106e+08,374.230414,374.230414,374.230414,1.077611e+05,1.053407e+05
min,2.651822e+02,2.508709e-01,30.000000,30.000000,511.000000,2.707252e+02,2.651822e+02
25%,8.586764e+03,2.748673e+06,608.250000,608.250000,1089.250000,8.747175e+03,8.586764e+03
50%,2.003280e+04,1.302106e+07,731.500000,731.500000,1212.500000,2.076504e+04,2.003280e+04
75%,5.737864e+04,7.407661e+07,1026.500000,1026.500000,1507.500000,5.876284e+04,5.737864e+04
max,1.031359e+06,9.052571e+09,1524.000000,1524.000000,2005.000000,1.072538e+06,1.031359e+06



Distribusi kategori layer_revi:
layer_revi
gambut    348
            2
Name: count, dtype: int64


## Data Quality Check

Memeriksa nilai kosong pada setiap kolom dataset.

In [33]:
# Menghitung missing values
missing_values = gdf_peatland.isnull().sum()

missing_summary = pd.DataFrame({
    "missing_count": missing_values,
    "missing_percentage": (
        missing_values / len(gdf_peatland) * 100
    )
})

display(missing_summary)

,missing_count,missing_percentage
Shape_Length,0,0.0
Shape_Area,0,0.0
OBJECTID_12,0,0.0
objectid_1,0,0.0
objectid,0,0.0
layer_revi,0,0.0
shape_leng,0,0.0
globalid,0,0.0
shape_Le_1,0,0.0
geometry,0,0.0


In [34]:
# Memeriksa duplikasi berdasarkan seluruh atribut
duplicate_count = gdf_peatland.duplicated().sum()

# Memeriksa duplikasi berdasarkan ID
duplicate_id = gdf_peatland["objectid"].duplicated().sum()

print("Duplikasi seluruh baris:", duplicate_count)
print("Duplikasi objectid:", duplicate_id)

Duplikasi seluruh baris: 0
Duplikasi objectid: 0


In [35]:
# Memeriksa geometri kosong
empty_geometry = gdf_peatland.geometry.is_empty.sum()

# Memeriksa geometri tidak valid
invalid_geometry = (~gdf_peatland.geometry.is_valid).sum()

# Memeriksa geometri null
null_geometry = gdf_peatland.geometry.isnull().sum()

print("Geometri kosong:", empty_geometry)
print("Geometri tidak valid:", invalid_geometry)
print("Geometri null:", null_geometry)

Geometri kosong: 0
Geometri tidak valid: 0
Geometri null: 0


In [36]:
# Melihat seluruh kategori beserta jumlahnya
category_distribution = (
    gdf_peatland["layer_revi"]
    .value_counts(dropna=False)
    .reset_index()
)

category_distribution.columns = [
    "category",
    "count"
]

display(category_distribution)

,category,count
0,gambut,348
1,,2


In [37]:
# Melihat batas koordinat dataset
print("Bounds dataset:")
print(gdf_peatland.total_bounds)

# Melihat rentang koordinat
minx, miny, maxx, maxy = gdf_peatland.total_bounds

print("\nLongitude:", minx, "sampai", maxx)
print("Latitude:", miny, "sampai", maxy)

Bounds dataset:
[108.85581731  -3.47796962 118.58359522   4.0767894 ]

Longitude: 108.85581730878893 sampai 118.58359521527423
Latitude: -3.477969622579607 sampai 4.076789396267116


## Category Inspection

Memeriksa nilai kategori lahan gambut untuk mengidentifikasi
nilai kosong atau kategori yang tidak terdefinisi.

In [38]:
# Memeriksa nilai kategori dalam representasi aslinya
for category in gdf_peatland["layer_revi"].unique():
    print(repr(category))

'gambut'
' '


## Spatial Boundary Integration

Memuat batas administrasi lima provinsi Kalimantan
sebagai acuan validasi cakupan geografis data gambut.

In [39]:
# Lokasi batas provinsi
boundary_path = Path(
    "../../data/processed/boundaries/kalimantan_provinces.geojson"
)

# Membaca batas provinsi
gdf_provinces = gpd.read_file(boundary_path)

print("Jumlah provinsi:", len(gdf_provinces))
print("CRS:", gdf_provinces.crs)

print("\nDaftar provinsi:")
print(gdf_provinces["wadmpr"].tolist())

Jumlah provinsi: 5
CRS: EPSG:4326

Daftar provinsi:
['Kalimantan Barat', 'Kalimantan Selatan', 'Kalimantan Tengah', 'Kalimantan Timur', 'Kalimantan Utara']


In [40]:
# Menyamakan CRS peatland dengan batas provinsi
if gdf_peatland.crs != gdf_provinces.crs:
    gdf_peatland = gdf_peatland.to_crs(gdf_provinces.crs)

print("CRS Peatland:", gdf_peatland.crs)
print("CRS Provinces:", gdf_provinces.crs)

CRS Peatland: EPSG:4326
CRS Provinces: EPSG:4326


In [41]:
# Spatial join menggunakan predicate intersects
peatland_intersection = gpd.sjoin(
    gdf_peatland,
    gdf_provinces[["wadmpr", "geometry"]],
    how="left",
    predicate="intersects"
)

# Menghitung jumlah fitur yang memiliki pasangan provinsi
matched = peatland_intersection[
    peatland_intersection["wadmpr"].notna()
]

unmatched = peatland_intersection[
    peatland_intersection["wadmpr"].isna()
]

print("Total baris hasil spatial join:", len(peatland_intersection))
print("Jumlah fitur yang beririsan:", matched.index.nunique())
print("Jumlah fitur tanpa pasangan provinsi:", unmatched.index.nunique())

print("\nDistribusi berdasarkan provinsi:")
print(matched["wadmpr"].value_counts())

Total baris hasil spatial join: 353
Jumlah fitur yang beririsan: 350
Jumlah fitur tanpa pasangan provinsi: 0

Distribusi berdasarkan provinsi:
wadmpr
Kalimantan Barat      178
Kalimantan Tengah     122
Kalimantan Timur       32
Kalimantan Utara       16
Kalimantan Selatan      5
Name: count, dtype: int64


## Multiple Province Intersection Check

Mengidentifikasi polygon gambut yang beririsan dengan
lebih dari satu wilayah administrasi provinsi.

In [42]:
# Menghitung jumlah pasangan provinsi untuk setiap polygon
province_match_count = (
    peatland_intersection
    .groupby(level=0)["wadmpr"]
    .nunique()
)

# Mengambil polygon yang memiliki lebih dari satu pasangan provinsi
multiple_matches = province_match_count[
    province_match_count > 1
]

print("Jumlah polygon dengan lebih dari satu provinsi:",
      len(multiple_matches))

print("\nDetail polygon:")
for idx in multiple_matches.index:
    print(
        "Index:", idx,
        "| Object ID:", gdf_peatland.loc[idx, "objectid"],
        "| Provinsi:",
        peatland_intersection.loc[
            peatland_intersection.index == idx, "wadmpr"
        ].unique().tolist()
    )

Jumlah polygon dengan lebih dari satu provinsi: 3

Detail polygon:
Index: 202 | Object ID: 1311 | Provinsi: ['Kalimantan Tengah', 'Kalimantan Barat']
Index: 210 | Object ID: 1341 | Provinsi: ['Kalimantan Tengah', 'Kalimantan Barat']
Index: 252 | Object ID: 1449 | Provinsi: ['Kalimantan Selatan', 'Kalimantan Tengah']


In [43]:
# Mengambil kategori yang hanya berisi spasi atau kosong
blank_category = gdf_peatland[
    gdf_peatland["layer_revi"].str.strip().eq("")
]

print("Jumlah kategori kosong:", len(blank_category))

display(
    blank_category[
        ["objectid", "layer_revi", "Shape_Area", "geometry"]
    ]
)

Jumlah kategori kosong: 2


,objectid,layer_revi,Shape_Area,geometry
348,2004,,2.620673e+08,"POLYGON ((114.4116 -1.64498, 114.41016 -1.6447..."
349,2005,,1.083582e+06,"POLYGON ((108.9842 0.54194, 108.98416 0.54198,..."


## Category Standardization

Melakukan standardisasi atribut kategori lahan gambut.
Nilai kosong atau hanya berisi spasi akan ditandai sebagai
'unknown' tanpa mengubah atribut sumber.

In [44]:
# Membuat salinan dataset
gdf_peatland_clean = gdf_peatland.copy()

# Membersihkan spasi pada atribut kategori
gdf_peatland_clean["layer_revi_clean"] = (
    gdf_peatland_clean["layer_revi"]
    .str.strip()
)

# Mengubah kategori kosong menjadi unknown
gdf_peatland_clean["layer_revi_clean"] = (
    gdf_peatland_clean["layer_revi_clean"]
    .replace("", "unknown")
)

# Memeriksa distribusi kategori setelah normalisasi
print("Distribusi kategori setelah cleaning:")
print(
    gdf_peatland_clean["layer_revi_clean"]
    .value_counts(dropna=False)
)

Distribusi kategori setelah cleaning:
layer_revi_clean
gambut     348
unknown      2
Name: count, dtype: int64


## Province Relationship Mapping

Membuat tabel relasi antara polygon gambut dan provinsi
berdasarkan hasil spatial intersection.

In [45]:
# Mengambil hubungan unik antara objectid dan provinsi
peatland_province = (
    peatland_intersection[
        ["objectid", "wadmpr"]
    ]
    .dropna()
    .drop_duplicates()
    .sort_values(["objectid", "wadmpr"])
    .reset_index(drop=True)
)

print("Jumlah relasi polygon-provinsi:", len(peatland_province))

display(peatland_province.head(10))

Jumlah relasi polygon-provinsi: 353


,objectid,wadmpr
0,511,Kalimantan Tengah
1,528,Kalimantan Tengah
2,538,Kalimantan Tengah
3,541,Kalimantan Tengah
4,547,Kalimantan Tengah
5,553,Kalimantan Tengah
6,566,Kalimantan Tengah
7,570,Kalimantan Tengah
8,592,Kalimantan Tengah
9,596,Kalimantan Tengah


## Province Relationship Summary

Menghitung jumlah polygon gambut unik yang beririsan
dengan setiap provinsi.

In [46]:
# Menghitung jumlah objectid unik per provinsi
province_summary = (
    peatland_province
    .groupby("wadmpr")["objectid"]
    .nunique()
    .reset_index(name="peatland_polygon_count")
)

display(
    province_summary.sort_values(
        "peatland_polygon_count",
        ascending=False
    )
)

,wadmpr,peatland_polygon_count
0,Kalimantan Barat,178
2,Kalimantan Tengah,122
3,Kalimantan Timur,32
4,Kalimantan Utara,16
1,Kalimantan Selatan,5


In [47]:
print("===== PEATLAND CLEANING VALIDATION =====")

print("Jumlah polygon awal:", len(gdf_peatland))
print("Jumlah polygon setelah cleaning:", len(gdf_peatland_clean))

print(
    "Kategori unknown:",
    (gdf_peatland_clean["layer_revi_clean"] == "unknown").sum()
)

print(
    "Jumlah objectid unik:",
    gdf_peatland_clean["objectid"].nunique()
)

print(
    "Jumlah relasi polygon-provinsi:",
    len(peatland_province)
)

print(
    "Jumlah provinsi:",
    peatland_province["wadmpr"].nunique()
)

===== PEATLAND CLEANING VALIDATION =====
Jumlah polygon awal: 350
Jumlah polygon setelah cleaning: 350
Kategori unknown: 2
Jumlah objectid unik: 350
Jumlah relasi polygon-provinsi: 353
Jumlah provinsi: 5


## Numeric Attribute Inspection

Memeriksa atribut numerik untuk mengidentifikasi
kolom yang memiliki nilai identik atau perbedaan nilai.

In [48]:
# Kolom numerik yang akan diperiksa
numeric_columns = [
    "Shape_Length",
    "shape_leng",
    "shape_Le_1",
    "Shape_Area"
]

# Statistik deskriptif
display(
    gdf_peatland_clean[numeric_columns].describe().T
)

,count,mean,std,min,25%,50%,75%,max
Shape_Length,350.0,5.529812e+04,1.053407e+05,265.182156,8.586764e+03,2.003280e+04,5.737864e+04,1.031359e+06
shape_leng,350.0,5.630060e+04,1.077611e+05,270.725159,8.747175e+03,2.076504e+04,5.876284e+04,1.072538e+06
shape_Le_1,350.0,5.529812e+04,1.053407e+05,265.182156,8.586764e+03,2.003280e+04,5.737864e+04,1.031359e+06
Shape_Area,350.0,1.418179e+08,5.890106e+08,0.250871,2.748673e+06,1.302106e+07,7.407661e+07,9.052571e+09


In [49]:
# Membandingkan kesamaan nilai antar kolom panjang
length_comparison = pd.DataFrame({
    "Shape_Length_vs_shape_leng": (
        gdf_peatland_clean["Shape_Length"]
        == gdf_peatland_clean["shape_leng"]
    ),
    "Shape_Length_vs_shape_Le_1": (
        gdf_peatland_clean["Shape_Length"]
        == gdf_peatland_clean["shape_Le_1"]
    ),
    "shape_leng_vs_shape_Le_1": (
        gdf_peatland_clean["shape_leng"]
        == gdf_peatland_clean["shape_Le_1"]
    )
})

print("Jumlah nilai yang identik:")
display(length_comparison.sum())

print("\nRata-rata selisih absolut:")
print(
    (gdf_peatland_clean["Shape_Length"] -
     gdf_peatland_clean["shape_leng"]).abs().mean()
)

Jumlah nilai yang identik:


Shape_Length_vs_shape_leng      0
Shape_Length_vs_shape_Le_1    346
shape_leng_vs_shape_Le_1        0
dtype: int64


Rata-rata selisih absolut:
1254.6373949873034


In [50]:
# Menghitung korelasi atribut numerik
correlation = (
    gdf_peatland_clean[numeric_columns]
    .corr()
)

display(correlation)

,Shape_Length,shape_leng,shape_Le_1,Shape_Area
Shape_Length,1.000000,0.999780,1.000000,0.857984
shape_leng,0.999780,1.000000,0.999780,0.864865
shape_Le_1,1.000000,0.999780,1.000000,0.857984
Shape_Area,0.857984,0.864865,0.857984,1.000000


## Length Attribute Difference Inspection

Mengidentifikasi polygon yang memiliki perbedaan nilai
antara Shape_Length dan shape_Le_1.

In [51]:
# Mencari baris dengan nilai panjang yang berbeda
length_mismatch = gdf_peatland_clean[
    gdf_peatland_clean["Shape_Length"]
    != gdf_peatland_clean["shape_Le_1"]
].copy()

# Menampilkan atribut yang relevan
display(
    length_mismatch[
        [
            "objectid",
            "Shape_Length",
            "shape_leng",
            "shape_Le_1"
        ]
    ]
)

print("Jumlah polygon berbeda:", len(length_mismatch))

,objectid,Shape_Length,shape_leng,shape_Le_1
163,1191,74571.383397,74790.156753,74571.383397
287,1800,219351.050199,225316.772541,219351.050199
296,1853,276687.680779,279165.275268,276687.680779
346,1998,594138.222872,592574.605606,594138.222872


Jumlah polygon berbeda: 4


In [52]:
# Membuat tabel perbandingan
length_difference = gdf_peatland_clean[
    ["objectid", "Shape_Length", "shape_leng", "shape_Le_1"]
].copy()

# Menghitung selisih absolut
length_difference["difference_1"] = (
    length_difference["Shape_Length"]
    - length_difference["shape_leng"]
).abs()

length_difference["difference_2"] = (
    length_difference["Shape_Length"]
    - length_difference["shape_Le_1"]
).abs()

# Menampilkan 10 perbedaan terbesar
display(
    length_difference.sort_values(
        "difference_1",
        ascending=False
    ).head(10)
)

,objectid,Shape_Length,shape_leng,shape_Le_1,difference_1,difference_2
261,1500,1.031359e+06,1.072538e+06,1.031359e+06,41178.609916,0.0
265,1552,4.593620e+05,4.807884e+05,4.593620e+05,21426.413093,0.0
260,1486,4.977099e+05,5.173972e+05,4.977099e+05,19687.297022,0.0
258,1483,3.710892e+05,3.880285e+05,3.710892e+05,16939.319518,0.0
219,1366,8.009196e+05,8.127966e+05,8.009196e+05,11877.024934,0.0
277,1660,9.199608e+04,1.034836e+05,9.199608e+04,11487.511960,0.0
234,1401,4.695279e+05,4.807989e+05,4.695279e+05,11270.969529,0.0
207,1328,3.523045e+05,3.618728e+05,3.523045e+05,9568.245322,0.0
275,1653,1.419847e+05,1.505355e+05,1.419847e+05,8550.771407,0.0
217,1363,1.107986e+05,1.186886e+05,1.107986e+05,7889.950083,0.0


## Floating Point Precision Inspection

Memeriksa perbedaan nilai atribut panjang dengan presisi
tinggi untuk mengetahui apakah perbedaannya hanya berasal
dari floating-point precision.

In [53]:
import numpy as np

# Mengambil polygon yang sebelumnya terdeteksi berbeda
precision_check = length_mismatch[
    ["objectid", "Shape_Length", "shape_leng", "shape_Le_1"]
].copy()

# Menghitung selisih dengan presisi tinggi
precision_check["difference"] = (
    precision_check["Shape_Length"]
    - precision_check["shape_Le_1"]
)

# Memeriksa apakah nilainya dianggap sama dengan toleransi
precision_check["is_close"] = np.isclose(
    precision_check["Shape_Length"],
    precision_check["shape_Le_1"],
    rtol=1e-9,
    atol=1e-8
)

# Menampilkan angka dengan presisi tinggi
pd.set_option("display.precision", 12)

display(precision_check)

,objectid,Shape_Length,shape_leng,shape_Le_1,difference,is_close
163,1191,74571.383396531019,74790.156752700001,74571.383396531033,-0.000000000015,True
287,1800,219351.050199183053,225316.772541000013,219351.050199183082,-0.000000000029,True
296,1853,276687.680778800917,279165.275268000027,276687.680778800335,0.000000000582,True
346,1998,594138.222872317419,592574.605606000056,594138.222872320912,-0.000000003492,True


In [54]:
# Membandingkan nilai dengan toleransi numerik
print(
    "Shape_Length dan shape_Le_1 dianggap sama:",
    np.isclose(
        gdf_peatland_clean["Shape_Length"],
        gdf_peatland_clean["shape_Le_1"],
        rtol=1e-9,
        atol=1e-8
    ).sum(),
    "dari",
    len(gdf_peatland_clean)
)

Shape_Length dan shape_Le_1 dianggap sama: 350 dari 350


## Final Numeric Attribute Validation

Memastikan kesamaan numerik antara Shape_Length dan
shape_Le_1 menggunakan toleransi floating-point.

In [55]:
# Memeriksa kesamaan numerik
same_length = np.isclose(
    gdf_peatland_clean["Shape_Length"],
    gdf_peatland_clean["shape_Le_1"],
    rtol=1e-9,
    atol=1e-8
)

print("Jumlah nilai yang dianggap sama:", same_length.sum())
print("Jumlah nilai yang berbeda:", (~same_length).sum())

Jumlah nilai yang dianggap sama: 350
Jumlah nilai yang berbeda: 0


## Save Processed Dataset

Menyiapkan direktori penyimpanan hasil preprocessing
data lahan gambut.

In [56]:
# Menentukan folder output
output_dir = Path("../../data/processed/peatland")

# Membuat folder jika belum tersedia
output_dir.mkdir(parents=True, exist_ok=True)

print("Output directory:", output_dir.resolve())

Output directory: D:\Project\karhutla-spatial-temporal-analysis\data\processed\peatland


In [57]:
# Lokasi output GeoJSON
peatland_output = (
    output_dir / "peatland_kalimantan_clean.geojson"
)

# Menyimpan dataset hasil cleaning
gdf_peatland_clean.to_file(
    peatland_output,
    driver="GeoJSON"
)

print("GeoJSON berhasil disimpan:")
print(peatland_output.resolve())

GeoJSON berhasil disimpan:
D:\Project\karhutla-spatial-temporal-analysis\data\processed\peatland\peatland_kalimantan_clean.geojson


In [58]:
# Lokasi output CSV
relationship_output = (
    output_dir / "peatland_province_relationship.csv"
)

# Menyimpan tabel relasi
peatland_province.to_csv(
    relationship_output,
    index=False
)

print("CSV relasi berhasil disimpan:")
print(relationship_output.resolve())

CSV relasi berhasil disimpan:
D:\Project\karhutla-spatial-temporal-analysis\data\processed\peatland\peatland_province_relationship.csv


## Processed Dataset Validation

Memastikan dataset hasil cleaning berhasil disimpan
dan dapat dibaca kembali tanpa kehilangan record.

In [59]:
# Membaca kembali GeoJSON
gdf_peatland_check = gpd.read_file(peatland_output)

# Membaca kembali CSV
df_relationship_check = pd.read_csv(relationship_output)

print("===== FINAL PEATLAND VALIDATION =====")

print("Jumlah polygon:", len(gdf_peatland_check))
print("Jumlah objectid unik:", gdf_peatland_check["objectid"].nunique())
print("CRS:", gdf_peatland_check.crs)

print("\nDistribusi kategori:")
print(gdf_peatland_check["layer_revi_clean"].value_counts())

print("\nJumlah relasi provinsi:", len(df_relationship_check))
print("Jumlah provinsi:", df_relationship_check["wadmpr"].nunique())

print("\nGeometry validity:")
print(gdf_peatland_check.geometry.is_valid.value_counts())

print("\nUkuran file:")
print("GeoJSON:", round(peatland_output.stat().st_size / 1024 / 1024, 2), "MB")
print("CSV:", round(relationship_output.stat().st_size / 1024, 2), "KB")

===== FINAL PEATLAND VALIDATION =====
Jumlah polygon: 350
Jumlah objectid unik: 350
CRS: EPSG:4326

Distribusi kategori:
layer_revi_clean
gambut     348
unknown      2
Name: count, dtype: int64

Jumlah relasi provinsi: 353
Jumlah provinsi: 5

Geometry validity:
True    350
Name: count, dtype: int64

Ukuran file:
GeoJSON: 11.07 MB
CSV: 8.04 KB
